# Ray Serve Demo

This notebook demonstrates how to train a model and serve it for inference simultaneously using Ray and Ray Serve.

In [1]:
# Import required libraries
import os
import time
import ray
import numpy as np
import requests
from ray import serve
from ray.serve.deployment import Deployment
from ray.serve._private.common import DeploymentStatus
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import threading
import json

## Initialize Ray and Load Data

In [2]:
# Initialize Ray
ray.init()

# Load a sample dataset (breast cancer dataset)
def load_data():
    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    return X_train, X_test, y_train, y_test

2025-05-07 19:18:41,583	INFO worker.py:1879 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
(ProxyActor pid=19414) INFO 2025-05-07 19:18:42,673 proxy 127.0.0.1 -- Proxy starting on node aee5b9cb2f3f1f675aee848079dcbbc4ea7b0a14cd3a16aaef518f49 (HTTP port: 8000).
(ProxyActor pid=19414) INFO 2025-05-07 19:18:42,688 proxy 127.0.0.1 -- Got updated endpoints: {}.
(ServeController pid=19410) INFO 2025-05-07 19:18:42,786 controller 19410 -- Deploying new version of Deployment(name='predictor', app='default') (initial target replicas: 1).
(ProxyActor pid=19414) INFO 2025-05-07 19:18:42,788 proxy 127.0.0.1 -- Got updated endpoints: {Deployment(name='predictor', app='default'): EndpointInfo(route='/predict', app_is_cross_language=False)}.
(ProxyActor pid=19414) INFO 2025-05-07 19:18:42,791 proxy 127.0.0.1 -- Started <ray.serve._private.router.SharedRouterLongPollClient object at 0x11948e6d0>.
(ServeController pid=19410) INFO 2025-05-07 19:18:42,888 controller 19410 -

(ServeReplica:default:predictor pid=19411) Initial model accuracy: 0.8465


(ServeReplica:default:predictor pid=19411) INFO 2025-05-07 19:18:43,854 default_predictor t9sd917r 54d0495f-7bf3-43b7-81d1-59b30ed6e540 -- POST /predict 200 3.6ms


(train_model pid=19409) Starting training iteration 0...


(ServeReplica:default:predictor pid=19411) INFO 2025-05-07 19:18:46,875 default_predictor t9sd917r e62edc14-fa5f-475c-be75-c865a656ffcf -- POST /predict 200 4.1ms


...

(ServeReplica:default:predictor pid=19411) INFO 2025-05-07 19:19:28,615 default_predictor t9sd917r 6f4053bb-76ba-4784-bcee-fea1cde4b04c -- CALL update_model OK 0.7ms
(ServeReplica:default:predictor pid=19411) INFO 2025-05-07 19:19:29,248 default_predictor t9sd917r 804848af-e224-4c8c-96e0-072625fad08f -- POST /predict 200 5.8ms
(ServeReplica:default:predictor pid=19411) INFO 2025-05-07 19:19:32,277 default_predictor t9sd917r 2e3d40d6-baa1-4a5a-8238-325f1f49af4e -- POST /predict 200 7.1ms
(ServeReplica:default:predictor pid=19411) INFO 2025-05-07 19:19:35,301 default_predictor t9sd917r 3891d0f4-fb69-46ce-a96b-bcc4584b1895 -- POST /predict 200 5.4ms


(train_model pid=19409) Starting training iteration 4...


(ServeReplica:default:predictor pid=19411) INFO 2025-05-07 19:19:38,324 default_predictor t9sd917r b3619d2b-dba8-4129-90eb-7bbf2c298ce5 -- POST /predict 200 6.8ms
(ServeReplica:default:predictor pid=19411) INFO 2025-05-07 19:19:41,353 default_predictor t9sd917r 0e32fd36-45d9-49bd-9803-251629d08b80 -- POST /predict 200 6.2ms


(ServeReplica:default:predictor pid=19411) Updating model to version 5 with accuracy 0.9649
(train_model pid=19409) Completed training iteration 4. Accuracy: 0.9649


(ServeReplica:default:predictor pid=19411) INFO 2025-05-07 19:19:41,689 default_predictor t9sd917r 0fa32d0c-1534-41d7-8222-a28ef3eecadb -- CALL update_model OK 0.6ms
(ServeController pid=19410) INFO 2025-05-07 19:19:44,448 controller 19410 -- Removing 1 replica from Deployment(name='predictor', app='default').
(ServeController pid=19410) INFO 2025-05-07 19:19:46,503 controller 19410 -- Replica(id='t9sd917r', deployment='predictor', app='default') is stopped.


## Define Model Training Function

In [3]:
@ray.remote
def train_model(iteration):
    print(f"Starting training iteration {iteration}...")

    # Load data
    X_train, X_test, y_train, y_test = load_data()

    # Train a simple model (we'll use RandomForest for demonstration)
    model = RandomForestClassifier(
        n_estimators=1 + (iteration * 5),  # Increase complexity with each iteration
        random_state=42
    )

    # Simulate longer training time
    time.sleep(5)

    # Train the model
    model.fit(X_train, y_train)

    # Evaluate the model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"Completed training iteration {iteration}. Accuracy: {accuracy:.4f}")

    return model, accuracy

## Define Model Predictor Class

In [4]:
@serve.deployment(name="predictor")
class ModelPredictor:
    def __init__(self):
        self.model = None
        self.model_version = 0
        self.accuracy = 0
        self.ready = False
        # Load initial model
        X, y = load_breast_cancer(return_X_y=True)
        self.model = RandomForestClassifier(n_estimators=1, random_state=42)
        self.model.fit(X[:100], y[:100])  # Train on a small subset for initial model
        
        # Calculate initial accuracy
        y_pred = self.model.predict(X[100:])  # Predict on the rest of the data
        self.accuracy = accuracy_score(y[100:], y_pred)
        print(f"Initial model accuracy: {self.accuracy:.4f}")
        
        self.ready = True

    async def update_model(self, model, accuracy, version):
        print(f"Updating model to version {version} with accuracy {accuracy:.4f}")
        self.model = model
        self.accuracy = accuracy
        self.model_version = version

    async def __call__(self, request):
        if not self.ready or self.model is None:
            return {"error": "Model not ready yet"}

        # Parse the request data
        try:
            input_data = await request.json()
            data = np.array(input_data["data"]).reshape(1, -1)
        except Exception as e:
            return {"error": str(e)}

        # Make prediction
        try:
            prediction = int(self.model.predict(data)[0])
            probability = float(self.model.predict_proba(data)[0][1])

            return {
                "prediction": prediction,
                "probability": probability,
                "model_version": self.model_version,
                "model_accuracy": float(self.accuracy)
            }
        except Exception as e:
            return {"error": f"Prediction error: {str(e)}"}

    async def status(self, request):
        return {
            "ready": self.ready,
            "model_version": self.model_version,
            "accuracy": float(self.accuracy)
        }

## Define Helper Functions for the Demo

In [5]:
def make_prediction():
    # Load a sample data point
    X, y = load_breast_cancer(return_X_y=True)
    sample = X[0].tolist()

    # Make a prediction request
    try:
        response = requests.post(
            "http://localhost:8000/predict",
            json={"data": sample}
        )
        result = response.json()
        print(f"Prediction result: {result}")
        print(f"True label: {y[0]}")
        return result
    except Exception as e:
        print(f"Error making prediction: {e}")
        return None

## Run the Demo

In [6]:
async def run_demo():
    print("Starting Ray Serve demo: Training and inference happening simultaneously")

    # Start Ray Serve
    serve.start()

    # Deploy the predictor
    predictor = serve.run(ModelPredictor.bind(), route_prefix="/predict")
    print("Model predictor deployed")

    # Wait for deployment to be ready
    while True:
        status = serve.status()
        if status.applications["default"].deployments["predictor"].status == DeploymentStatus.HEALTHY:
            break
        print("Waiting for deployment to be ready...")
        time.sleep(1)

    print("Predictor is ready to serve requests")

    # Run inference in a loop
    def inference_loop():
        print("Starting inference loop...")
        for i in range(20):
            print(f"\nInference request {i + 1}:")
            result = make_prediction()
            time.sleep(3)  # Make a request every 3 seconds

        print("Inference loop complete!")

    # Start inference in a separate thread
    inference_thread = threading.Thread(target=inference_loop)
    inference_thread.daemon = True
    inference_thread.start()

    # Start training in the background and update the model as training progresses
    model_futures = []
    for i in range(5):  # Train 5 iterations of the model
        # Start a training task
        model_future = train_model.remote(i)
        model_futures.append(model_future)

        # Wait for this training iteration to complete
        model, accuracy = ray.get(model_future)

        # Get a handle to the deployment and update the model
        handle = serve.get_deployment_handle("predictor", app_name="default")
        await handle.update_model.remote(model, accuracy, i + 1)

        # Small delay between training iterations
        if i < 4:  # Don't sleep after the last iteration
            time.sleep(8)

    # Wait for the inference thread to complete
    inference_thread.join()

    print("Demo completed!")

    # Shutdown Ray Serve
    serve.shutdown()

    # Shutdown Ray
    ray.shutdown()

## Execute the Demo

In [7]:
import asyncio
await run_demo()

Starting Ray Serve demo: Training and inference happening simultaneously


INFO 2025-05-07 19:18:42,719 serve 19246 -- Started Serve in namespace "serve".
INFO 2025-05-07 19:18:42,720 serve 19246 -- Connecting to existing Serve app in namespace "serve". New http options will not be applied.
INFO 2025-05-07 19:18:43,834 serve 19246 -- Application 'default' is ready at http://127.0.0.1:8000/predict.


Model predictor deployed
Predictor is ready to serve requests
Starting inference loop...

Inference request 1:
Prediction result: {'prediction': 0, 'probability': 0.0, 'model_version': 0, 'model_accuracy': 0.8464818763326226}
True label: 0

Inference request 2:
Prediction result: {'prediction': 0, 'probability': 0.0, 'model_version': 0, 'model_accuracy': 0.8464818763326226}
True label: 0


INFO 2025-05-07 19:18:49,442 serve 19246 -- Started <ray.serve._private.router.SharedRouterLongPollClient object at 0x14b053cd0>.



Inference request 3:
Prediction result: {'prediction': 0, 'probability': 0.0, 'model_version': 1, 'model_accuracy': 0.9385964912280702}
True label: 0

Inference request 4:
Prediction result: {'prediction': 0, 'probability': 0.0, 'model_version': 1, 'model_accuracy': 0.9385964912280702}
True label: 0

Inference request 5:
Prediction result: {'prediction': 0, 'probability': 0.0, 'model_version': 1, 'model_accuracy': 0.9385964912280702}
True label: 0

Inference request 6:
Prediction result: {'prediction': 0, 'probability': 0.0, 'model_version': 1, 'model_accuracy': 0.9385964912280702}
True label: 0

Inference request 7:
Prediction result: {'prediction': 0, 'probability': 0.0, 'model_version': 1, 'model_accuracy': 0.9385964912280702}
True label: 0

Inference request 8:
Prediction result: {'prediction': 0, 'probability': 0.0, 'model_version': 2, 'model_accuracy': 0.9473684210526315}
True label: 0

Inference request 9:
Prediction result: {'prediction': 0, 'probability': 0.0, 'model_version'